<a href="https://colab.research.google.com/github/christinakoump/Parkinson-s-Disease-Severity-Assessment-on-the-UPDRS-Using-Text-Embeddings-Generated-from-Speech/blob/main/whisper_mxbai_SVR_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
!nvidia-smi


## Dataset Structure Overview

This step scans the PC-GITA dataset directory to analyze its hierarchical structure. It identifies all subfolders and counts the number of `.wav` audio files in each directory. This is used to verify dataset completeness and understand the organization of speech tasks before further processing.

In [ ]:
import os
from collections import defaultdict
root_path = "/media/blue/ckoumpo/pcgita/PC-GITA_per_task_44100Hz/"

folder_structure = defaultdict(dict)
####it was list

for dirpath, dirnames, filenames in os.walk(root_path):
    rel_path = os.path.relpath(dirpath, root_path)

    if rel_path == ".":
        continue

    subfolders = [name for name in os.listdir(dirpath) if os.path.isdir(os.path.join(dirpath, name))]
    wav_count = sum(1 for f in filenames if f.lower().endswith('.wav'))

    folder_structure[rel_path] = {
        "Subfolders": subfolders,
        "WAV files": wav_count
    }

for path, info in folder_structure.items():
    print(f" {path}")
    if info["Subfolders"]:
        print(f"   ├─ Subfolders: {', '.join(info['Subfolders'])}")
    if info["WAV files"] > 0:
        print(f"   └─ WAV files: {info['WAV files']}")


## WAV File Indexing and Label Extraction

This step iterates through the PC-GITA dataset to locate all `.wav` audio files and build a structured index. For each file, it extracts the task type, speaker ID, and full file path. It also assigns a clinical label (PD, HC, or Unknown) based on directory keywords such as “parkinson”, “control”, or “hc”. The final output is saved as a CSV file for downstream processing and analysis.

In [ ]:
import os
import pandas as pd

root_path = "/media/blue/ckoumpo/pcgita/PC-GITA_per_task_44100Hz/"
save_csv = "/media/blue/ckoumpo/pcgita/PCGITA_RESULTS/pcgita_file_index.csv"

folder_records = []

for dirpath, _, filenames in os.walk(root_path):
    for fname in filenames:
        if not fname.lower().endswith(".wav"):
            continue

        full_path = os.path.join(dirpath, fname)
        relative_path = os.path.relpath(full_path, root_path)
        parts = relative_path.split(os.sep)

        task = parts[0]
        speaker_id = fname.split("_")[0]

        path_lower = [p.lower() for p in parts]
        if any(x in p for p in path_lower for x in ["patologica", "parkinson", "pd"]):
            label = "PD"
        elif any(x in p for p in path_lower for x in ["control", "hc", "normal"]):
            label = "HC"
        else:
            label = "Unknown"

        folder_records.append({
            "Task": task,
            "Label": label,
            "Speaker_ID": speaker_id,
            "Relative Path": relative_path,
            "Full Path": full_path,
        })

df = pd.DataFrame(folder_records)
os.makedirs(os.path.dirname(save_csv), exist_ok=True)
df.to_csv(save_csv, index=False)

print(f"Indexed WAV files: {len(df)}")
df.head()


## Metadata Integration and Cleaning

This step loads clinical metadata from the PC-GITA Excel file and merges it with the previously created audio file index. Speaker IDs are standardized to ensure correct alignment between datasets. The merged dataset is then filtered to keep only valid Parkinson’s Disease (PD) and Healthy Control (HC) samples, while removing records with missing clinical information such as UPDRS score, age, gender, or disease stage. The final cleaned dataset is saved as a CSV file for downstream modeling.

In [ ]:
import pandas as pd
import openpyxl

xlsx_path = "/media/blue/ckoumpo/pcgita/PC-GITA_per_task_44100Hz/Copia de PCGITA_metadata.xlsx"
file_index_path = "/media/blue/ckoumpo/pcgita/PCGITA_RESULTS/pcgita_file_index.csv"
output_csv = "/media/blue/ckoumpo/pcgita/PCGITA_RESULTS/pcgita_metadata_cleaned.csv"

df_meta = pd.read_excel(xlsx_path)
df_meta.columns = df_meta.columns.str.strip().str.title()

df_meta = df_meta.rename(columns={
    "Recoding Original Name": "Speaker_ID",
    "Updrs": "UPDRS_total",
    "H/Y": "Hoehn_Yahr",
    "Sex": "Gender",
    "Age": "Age",
    "Time After Diagnosis": "After_Diagnosis"
})

df_index = pd.read_csv(file_index_path)

df_meta["Speaker_ID"] = df_meta["Speaker_ID"].astype(str).str.strip()
df_index["Speaker_ID"] = df_index["Speaker_ID"].astype(str).str.strip()

df_merged = pd.merge(df_index, df_meta, on="Speaker_ID", how="left")

df_merged = df_merged[df_merged["Label"].isin(["PD", "HC"])]
df_merged = df_merged[df_merged["Speaker_ID"].notna()]
df_merged = df_merged[df_merged["UPDRS_total"].notna()]

df_cleaned = df_merged.dropna(subset=["Age", "Gender", "Hoehn_Yahr", "After_Diagnosis"])

df_cleaned.to_csv(output_csv, index=False)
df_cleaned.head()


## Whisper Model Initialization

This step loads the pre-trained Whisper large-v2 model for automatic speech recognition. The model is configured to run on GPU if available, otherwise it falls back to CPU.

In [ ]:
import whisper
import torchaudio
import torch
#from spellchecker import SpellChecker
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Whisper large-v2
whisper_model = whisper.load_model("large-v2")
#spell = SpellChecker()

## Audio Preprocessing and Chunking

This function loads an audio file and prepares it for transcription by Whisper. The audio is converted to mono, resampled to 16 kHz if necessary, and normalized. It is then segmented into fixed-length overlapping chunks to improve transcription accuracy on long recordings. Each chunk preserves temporal continuity using overlap to avoid cutting words at segment boundaries.

In [ ]:
def preprocess_audio(file_path, chunk_length=30, overlap=5):
  #30 seconds chunk_lenght
  #5 seconds overlap

    waveform, sr = torchaudio.load(file_path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True) # mean, 1 dimension only, single channel

    # Resample if needed
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    # Normalize
    waveform = waveform / waveform.abs().max()

    waveform = waveform.squeeze().numpy()  #Removes the channel dimension → 1D array of length = total samples


    # Chunking
    chunks = []
    samples_per_chunk = chunk_length * 16000 #Converts the desired chunk length from seconds to samples
    samples_overlap = overlap * 16000 #Ensures consecutive chunks have shared audio to prevent cutting off words
    start = 0  #Sets the starting index for the first chunk

    while start < len(waveform):  #Keep cutting pieces until we reach the end
        end = start + samples_per_chunk #Where this piece ends.
        chunk = waveform[start:end] #Take the piece from start to end,The code slices this 1D .wav array into pieces/chunks
        chunks.append(chunk)
        start += samples_per_chunk - samples_overlap #Move the starting point forward for the next piece, keeping the overlap.

    return chunks

In [ ]:
import pandas as pd

df = pd.read_csv("/media/blue/ckoumpo/pcgita/PCGITA_RESULTS/pcgita_metadata_cleaned.csv")

print("Total audio files:", len(df))
df.head()

## Speech-to-Text Transcription (Whisper)

This step converts audio recordings into text using the pre-trained Whisper large-v2 model. Each audio file is first split into overlapping chunks to handle long recordings. Each chunk is transcribed individually in Spanish, and the partial results are concatenated into a full transcript. The final transcripts are stored in the dataset for downstream embedding and prediction tasks.

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def transcribe_audio_spanish(file_path):

    chunks = preprocess_audio(file_path)
    texts = []

    for chunk in chunks:
        result = whisper_model.transcribe(
            np.array(chunk),
            task="transcribe",  # keep original language
            language="es",      # Spanish
            temperature=0.0
        )
        if result["text"]:  # skip None or empty
            texts.append(result["text"])

    if len(texts) == 0:
        return ""  # empty string if no chunks

    transcript = " ".join(texts).strip()

    return transcript  # just raw Spanish transcript for now

# Apply transcription
df["Transcript"] = df["Full Path"].progress_apply(transcribe_audio_spanish)

# Save updated CSV
df.to_csv("/media/blue/ckoumpo/pcgita/PCGITA_RESULTS/all_transcripts_improved_spanish.csv", index=False)

## Sentence Embedding Model Initialization

This step loads a pre-trained SentenceTransformer model to convert transcribed speech into dense semantic embeddings. These embeddings capture high-level linguistic and semantic information from the transcripts for use in the regression model.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedder = SentenceTransformer(
    "mixedbread-ai/mxbai-embed-large-v1",
    device=device
)

## Text Embedding Extraction

This function converts cleaned speech transcripts into dense vector representations using the SentenceTransformer model. It filters out invalid or empty texts, applies a consistent prompt prefix to improve task conditioning, and encodes the valid transcripts into normalized embeddings. If no valid text is available, the function returns `None`.

In [ ]:
import numpy as np

def get_embeddings(texts):
    cleaned = [
        "Parkinson speech severity prediction: " + t
        for t in texts
        if isinstance(t, str) and len(t.strip()) > 2
    ]

    if len(cleaned) == 0:
        return None

    return embedder.encode(
        cleaned,
        batch_size=16,
        normalize_embeddings=True
    )

## L2 Normalization

This function applies L2 normalization to a vector to ensure unit length. It computes the Euclidean norm and divides the vector by its magnitude. If the norm is zero, the original vector is returned unchanged. This step helps stabilize feature representations before model training.

In [ ]:
import numpy as np

def l2_normalize(vec):
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm

## Speaker-Level Embedding Aggregation

This step aggregates transcript-level embeddings into a single representation per speaker. Only Parkinson’s Disease (PD) samples are used. For each speaker, multiple utterance embeddings are computed, then combined using median pooling to improve robustness against outliers. The resulting speaker embedding is L2-normalized. The target variable (UPDRS score) is extracted per speaker to form the regression dataset.

In [ ]:
speaker_embeddings = []
y = []

df_pd = df[df["Label"] == "PD"]

for speaker, group in df_pd.groupby("Speaker_ID"):

    embeddings = get_embeddings(group["Transcript"].tolist())
    if embeddings is None:
        continue

    # robust aggregation
    embedding = np.median(embeddings, axis=0)

    # L2 normalization
    embedding = l2_normalize(embedding)

    speaker_embeddings.append(embedding)
    y.append(group["UPDRS_total"].iloc[0])

X = np.vstack(speaker_embeddings)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

## Model Training and Evaluation (SVR with LOOCV)

This step trains a Support Vector Regression (SVR) model to predict Parkinson’s disease severity (UPDRS scores) from speaker-level speech embeddings. A Leave-One-Out Cross-Validation (LOOCV) strategy is used due to the small dataset size. Each fold performs hyperparameter optimization using GridSearchCV over multiple SVR kernels (RBF, linear, sigmoid) and optional PCA dimensionality reduction. Performance is evaluated using RMSE, MAE, and R² metrics, and results are aggregated across all folds to identify the best-performing model configuration.

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import LeaveOneOut, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

loo = LeaveOneOut()

absolute_errors = []
results = []

for i, (train_idx, test_idx) in enumerate(loo.split(X)):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', 'passthrough'),
        ('svr', SVR())
    ])

    param_grid = [

        # ----------------------------------
        # NO PCA + RBF
        # ----------------------------------
        {
            'pca': ['passthrough'],
            'svr__kernel': ['rbf'],
            'svr__C': [0.1, 1, 10, 100, 200],
            'svr__gamma': ['scale', 1e-4, 1e-3, 1e-2],
            'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
        },

        # ----------------------------------
        # NO PCA + Linear
        # ----------------------------------
        {
            'pca': ['passthrough'],
            'svr__kernel': ['linear'],
            'svr__C': [0.1, 1, 10, 100, 200],
            'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
        },

        # ----------------------------------
        # PCA + RBF
        # ----------------------------------
        {
            'pca': [PCA()],
            'pca__n_components': [5, 10, 20, 30],
            'svr__kernel': ['rbf'],
            'svr__C': [0.1, 1, 5, 10, 100, 200],
            'svr__gamma': ['scale', 1e-4, 1e-3, 1e-2],
            'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
        },

        # ----------------------------------
        # PCA + Linear
        # ----------------------------------
        {
            'pca': [PCA()],
            'pca__n_components': [5, 10, 20, 30],
            'svr__kernel': ['linear'],
            'svr__C': [0.1, 1, 5, 10, 100, 200],
            'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
        },

        # ----------------------------------
        # PCA + Sigmoid
        # ----------------------------------
        {
            'pca': [PCA()],
            'pca__n_components': [5, 10, 20, 30],
            'svr__kernel': ['sigmoid'],
            'svr__C': [0.1, 1, 5, 10, 100],
            'svr__gamma': [1e-4, 1e-3, 1e-2],
            'svr__coef0': [0, 0.1, 1],
            'svr__epsilon': [0.01, 0.1, 0.5, 1.0]
        }
    ]

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        refit=True
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    y_pred = best_model.predict(X_test)

    abs_error = np.abs(y_test[0] - y_pred[0])
    squared_error = (y_test[0] - y_pred[0]) ** 2

    absolute_errors.append(abs_error)

    results.append({
        "fold": i,
        "true_y": y_test[0],
        "pred_y": y_pred[0],
        "abs_error": abs_error,
        "squared_error": squared_error,
        "best_cv_mse": -grid.best_score_,
        **grid.best_params_
    })

# ==================================
# RESULTS DATAFRAME
# ==================================

df_results = pd.DataFrame(results)
df_results["kernel"] = df_results["svr__kernel"]

# ==================================
# PCA SUMMARY
# ==================================

if "pca__n_components" in df_results.columns:

    print("\nPCA selections:")
    print(
        df_results["pca__n_components"]
        .value_counts(dropna=False)
        .sort_index()
    )

# ==================================
# KERNEL PERFORMANCE SUMMARY
# ==================================

kernel_stats = []

for kernel in sorted(df_results["kernel"].unique()):

    subset = df_results[df_results["kernel"] == kernel]

    rmse_kernel = np.sqrt(
        mean_squared_error(
            subset["true_y"],
            subset["pred_y"]
        )
    )

    mae_kernel = mean_absolute_error(
        subset["true_y"],
        subset["pred_y"]
    )

    r2_kernel = r2_score(
        subset["true_y"],
        subset["pred_y"]
    )

    se_kernel = (
        subset["abs_error"].std(ddof=1)
        / np.sqrt(len(subset))
        if len(subset) > 1 else np.nan
    )

    kernel_stats.append({
        "kernel": kernel,
        "times_chosen": len(subset),
        "RMSE": rmse_kernel,
        "MAE": mae_kernel,
        "R2": r2_kernel,
        "SE": se_kernel,
        "mean_abs_error": subset["abs_error"].mean(),
        "max_abs_error": subset["abs_error"].max(),
        "min_abs_error": subset["abs_error"].min()
    })

kernel_summary = pd.DataFrame(kernel_stats)

print("\n")
print("=" * 70)
print("KERNEL PERFORMANCE")
print("=" * 70)

print(
    kernel_summary
    .sort_values("RMSE", ascending=True)
    .round(4)
)

# ==================================
# BEST KERNEL
# ==================================

best_kernel_row = kernel_summary.loc[
    kernel_summary["RMSE"].idxmin()
]

print("\n")
print("=" * 70)
print("BEST KERNEL")
print("=" * 70)

print(
    f"Kernel: {best_kernel_row['kernel']}\n"
    f"RMSE: {best_kernel_row['RMSE']:.4f}\n"
    f"MAE: {best_kernel_row['MAE']:.4f}\n"
    f"R²: {best_kernel_row['R2']:.4f}\n"
    f"SE: {best_kernel_row['SE']:.4f}\n"
    f"Chosen: {int(best_kernel_row['times_chosen'])} folds"
)

# ==================================
# OVERALL PERFORMANCE
# ==================================

overall_rmse = np.sqrt(
    mean_squared_error(
        df_results["true_y"],
        df_results["pred_y"]
    )
)

overall_mae = mean_absolute_error(
    df_results["true_y"],
    df_results["pred_y"]
)

overall_r2 = r2_score(
    df_results["true_y"],
    df_results["pred_y"]
)

overall_se = (
    np.std(df_results["abs_error"], ddof=1)
    / np.sqrt(len(df_results))
)

print("\n")
print("=" * 70)
print("OVERALL PERFORMANCE")
print("=" * 70)

print(f"R²   : {overall_r2:.4f}")
print(f"RMSE : {overall_rmse:.4f}")
print(f"MAE  : {overall_mae:.4f}")
print(f"SE   : {overall_se:.4f}")

# ==================================
# DATAFRAME PREVIEW
# ==================================

df_results.head()

In [ ]:
# ==================================
# SCATTER PLOT: True vs Predicted
# ==================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

plt.figure(figsize=(6, 6))

# Scatter points
plt.scatter(
    df_results["true_y"],
    df_results["pred_y"],
    color="black",
    alpha=0.7
)

# Perfect prediction line
min_val = min(df_results["true_y"].min(), df_results["pred_y"].min())
max_val = max(df_results["true_y"].max(), df_results["pred_y"].max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    "r--",
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("True UPDRS")
plt.ylabel("Predicted UPDRS")

plt.title(
    f"LOOCV Predictions vs True Values\n"
    f"R² = {overall_r2:.3f}   "
    f"RMSE = {overall_rmse:.2f}   "
    f"MAE = {overall_mae:.2f}"
)

plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.gca().set_aspect('equal', adjustable='box')

plt.legend()
plt.tight_layout()
plt.show()